<a href="https://colab.research.google.com/github/nishaa25n-ai/self-healing-sentiment-classifier/blob/main/MLPRO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv('Dataset-SA.csv',on_bad_lines='skip',engine='python')
df.head()

,product_name,product_price,Rate,Review,Summary,Sentiment
0,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,5,super!,great cooler excellent air flow and for this p...,positive
1,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,5,awesome,best budget 2 fit cooler nice cooling,positive
2,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,3,fair,the quality is good but the power of air is de...,positive
3,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,1,useless product,very bad product its a only a fan,negative
4,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,3,fair,ok ok product,neutral


In [ ]:
df.info()
df['Sentiment'].value_counts()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205052 entries, 0 to 205051
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   product_name   205052 non-null  object
 1   product_price  205052 non-null  object
 2   Rate           205052 non-null  object
 3   Review         180388 non-null  object
 4   Summary        205041 non-null  object
 5   Sentiment      205052 non-null  object
dtypes: object(6)
memory usage: 9.4+ MB


,0
product_name,0
product_price,0
Rate,0
Review,24664
Summary,11
Sentiment,0


In [ ]:
df = df.dropna(subset=['Review'])
print(df.shape)
print(df['Sentiment'].value_counts())

(180388, 6)
Sentiment
positive    147176
negative     24401
neutral       8811
Name: count, dtype: int64


In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import pandas as pd
df = pd.read_csv('Dataset-SA.csv',on_bad_lines='skip',engine='python')
df = df.dropna(subset=['Review'])

In [ ]:
sample_text = df['Review'].iloc[0]
print(sample_text)

embedding = model.encode(sample_text)
print(embedding.shape)
print(embedding[:10])


super!
(384,)
[-0.1339524  -0.01200065 -0.03259418 -0.00941652 -0.04966565  0.03366148
  0.0469831  -0.06193299 -0.12583692  0.00529812]


In [ ]:
df_sample = df.sample(n=5000, random_state=42).reset_index(drop=True)
print(df_sample.shape)
print(df_sample['Sentiment'].value_counts())

(5000, 6)
Sentiment
positive    4139
negative     609
neutral      252
Name: count, dtype: int64


In [ ]:
X = model.encode(df_sample['Review'].tolist(), show_progress_bar=True)
print(X.shape)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

(5000, 384)


In [ ]:
y = df_sample['Sentiment']
print(y.value_counts())

Sentiment
positive    4139
negative     609
neutral      252
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_tarin, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

print("Training Data:",X_train.shape)
print("Training Data:",X_test.shape)

Training Data: (4000, 384)
Training Data: (1000, 384)


In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_tarin)

print("Model Training Complete!")

Model Training Complete!


In [ ]:
y_pred = clf.predict(X_test)

print(y_pred[:10])
print(y_test[:10].values)

['positive' 'positive' 'positive' 'positive' 'positive' 'positive'
 'positive' 'positive' 'positive' 'negative']
['positive' 'positive' 'positive' 'positive' 'positive' 'positive'
 'positive' 'positive' 'positive' 'negative']


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:",accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.911

Classification Report:
              precision    recall  f1-score   support

    negative       0.86      0.78      0.82       116
     neutral       0.00      0.00      0.00        57
    positive       0.92      0.99      0.95       827

    accuracy                           0.91      1000
   macro avg       0.59      0.59      0.59      1000
weighted avg       0.86      0.91      0.88      1000


Confusion Matrix:
[[ 91   0  25]
 [  8   0  49]
 [  7   0 820]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
clf_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')
clf_balanced.fit(X_train, y_tarin)

y_pred_balanced = clf_balanced.predict(X_test)

print("Accuarcy:", accuracy_score(y_test, y_pred_balanced))
print("\nClassification Report:")
print(classification_report(y_test , y_pred_balanced, zero_division=0))

Accuarcy: 0.771

Classification Report:
              precision    recall  f1-score   support

    negative       0.87      0.78      0.82       116
     neutral       0.11      0.42      0.18        57
    positive       0.96      0.79      0.87       827

    accuracy                           0.77      1000
   macro avg       0.65      0.66      0.62      1000
weighted avg       0.90      0.77      0.82      1000



In [ ]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment ("sentiment-classification")

with mlflow.start_run():
  mlflow.log_param("model", "Logistic Regresssion")
  mlflow.log_param("class_weight", "balanced")
  mlflow.log_param("max_iter",1000)

  mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred_balanced))

  mlflow.sklearn.log_model(clf_balanced, "model")

  print("MLFlow run complete! Experiment tracked")

2026/09/28 04:15:55 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/28 04:15:55 INFO mlflow.store.db.utils: Updating database tables
2026/09/28 04:16:01 INFO mlflow.tracking.fluent: Experiment with name 'sentiment-classification' does not exist. Creating a new experiment.
2026/09/28 04:16:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


MLFlow run complete! Experiment tracked


In [ ]:

!pip install fastapi uvicorn pyngrok nest-asyncio


In [ ]:
import joblib

joblib.dump(clf_balanced, 'sentiment_model.pkl')
print("Model saved! ")

Model saved! 


In [ ]:
%%writefile app.py
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
from sentence_transformers import SentenceTransformer

app = FastAPI()

clf = joblib.load('sentiment_model.pkl')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

class ReviewInput(BaseModel):
    text: str

@app.get("/")
def home():
    return {"message": "Sentiment Classification API is running!"}

@app.post("/predict")
def predict_sentiment(review: ReviewInput):
    embedding = embed_model.encode([review.text])
    prediction = clf.predict(embedding)
    return {"review": review.text, "sentiment": prediction[0]}

Writing app.py


In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("3JCyiKgnG8AFuPMP2VxzTNi86uy_6mq4sUSYb6HVVCMoBNP68")

In [ ]:
import threading
import uvicorn
from app import app

def run_api():
  uvicorn.run(app, host="0.0.0.0", port=8000)

  thread = threading.Thread(target=run_api)
  thread.start()

  print("API starting... wait")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("3JCyiKgnG8AFuPMP2VxzTNi86uy_6mq4sUSYb6HVVCMoBNP68")
ngrok_tunnel = ngrok.connect(8000)
print('Public URL:', ngrok_tunnel.public_url)

Public URL: https://aspire-record-tidy.ngrok-free.dev


In [ ]:
import threading
import time
import uvicorn
from app import app

def run_api():
    try:
        uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")
    except Exception as e:
        print("SERVER ERROR:", e)

thread = threading.Thread(target=run_api, daemon=True)
thread.start()

time.sleep(5)  # server ko start hone ka time do
print("Thread alive:", thread.is_alive())

INFO:     Started server process [2739]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Thread alive: True


In [ ]:
from pyngrok import ngrok

ngrok_tunnel = ngrok.connect(8000)
print('Public URL:', ngrok_tunnel.public_url)

Public URL: https://aspire-record-tidy.ngrok-free.dev


In [ ]:
!cat app.py

from fastapi import FastAPI
from pydantic import BaseModel
import joblib
from sentence_transformers import SentenceTransformer

app = FastAPI()

clf = joblib.load('sentiment_model.pkl')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

class ReviewInput(BaseModel):
    text: str

@app.get("/")
def home():
    return {"message": "Sentiment Classification API is running!"}

@app.post("/predict")
def predict_sentiment(review: ReviewInput):
    embedding = embed_model.encode([review.text])
    prediction = clf.predict(embedding)
    return {"review": review.text, "sentiment": prediction[0]}


In [ ]:
from scipy.stats import ks_2samp
import numpy as np



train_embedding_mean = X_train.mean(axis=1)

new_sample_texts = [
    "this is a completely different kind of review with new vocabulary",
    "totally unrealted content about something else entirely",
    "random new text that model has never seen before"
]

new_embeddings = model.encode(new_sample_texts)
new_embedding_mean = new_embeddings.mean(axis=1)

statistic, p_value = ks_2samp(train_embedding_mean , new_embedding_mean)

print("KS Statistic:",statistic)
print("P-value:",p_value)

if p_value < 0.05:
  print("DRIFT DETECTED !")
else:
  print("No significant drift detected.")


KS Statistic: 0.97425
P-value: 3.611829771870633e-05
DRIFT DETECTED !


In [ ]:
def check_drift(train_data, new_data, threshold= 0.05):
  train_mean = train_data.mean(axis=1)
  new_mean = new_data.mean(axis=1)

  statistic, p_value = ks_2samp(train_mean , new_mean)

  drift_detected = p_value < threshold

  result = {
      "ks_statistic": statistic,
      "p_value":p_value,
      "drift_detected":drift_detected
  }

  if drift_detected:
    print(f"DRIFT DETECTED ! (p-value : {p_value:.6f})")
  else:
    print(f"No drift detected (p-value : {p_value:.6f})")

  return result



In [ ]:
result = check_drift(X_train, new_embeddings)
print(result)

DRIFT DETECTED ! (p-value : 0.000036)
{'ks_statistic': np.float64(0.97425), 'p_value': np.float64(3.611829771870633e-05), 'drift_detected': np.True_}


In [ ]:
%%writefile requirements.txt
fastapi
uvicorn
scikit-learn
sentence-transformers
joblib
pandas
numpy
scipy

Writing requirements.txt


In [ ]:
%%writefile Dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

RUN python -c "from sentence_transformers import SentenceTransformer; SentenceTransformer('all-MiniLM-L6-v2')"

COPY app.py .
COPY sentiment_model.pkl .
COPY X_train_ref.npy .
COPY y_train_ref.npy .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

Overwriting Dockerfile


In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

np.save('X_train_ref.npy', X_train)
np.save('y_train_ref.npy', y_train.values)
print(X_train.shape, y_train.shape)

(4000, 384) (4000,)


In [ ]:
import numpy as np
np.save('X_train_ref.npy', X_train)
np.save('y_train_ref.npy', y_train.values)

In [5]:
%%writefile app.py
from fastapi import FastAPI, BackgroundTasks
from pydantic import BaseModel
import joblib, threading
import numpy as np
from scipy.stats import ks_2samp
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer

app = FastAPI()
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
clf = joblib.load('sentiment_model.pkl')
X_ref = np.load('X_train_ref.npy')
y_ref = np.load('y_train_ref.npy', allow_pickle=True)

recent_X = []
fb_X, fb_y = [], []
state = {"model_version": 1, "retrain_count": 0}
lock = threading.Lock()

WINDOW = 200
MIN_FEEDBACK = 50

def drift_fraction(ref, new):
    drifted = sum(
        ks_2samp(ref[:, i], new[:, i]).pvalue < 0.05
        for i in range(ref.shape[1])
    )
    return drifted / ref.shape[1]

def drift_detected(ref, new, frac_threshold=0.15):
    return drift_fraction(ref, new) > frac_threshold

def maybe_heal():
    global clf, X_ref, y_ref
    if len(recent_X) < WINDOW or len(fb_X) < MIN_FEEDBACK:
        return
    new = np.array(recent_X[-WINDOW:])
    if not drift_detected(X_ref, new):
        return
    with lock:
        X = np.vstack([X_ref, np.array(fb_X)])
        y = np.concatenate([y_ref, np.array(fb_y)])
        new_clf = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X, y)
        joblib.dump(new_clf, 'sentiment_model.pkl')
        clf = new_clf
        X_ref, y_ref = X, y
        recent_X.clear(); fb_X.clear(); fb_y.clear()
        state["model_version"] += 1
        state["retrain_count"] += 1

class ReviewInput(BaseModel):
    text: str

class Feedback(BaseModel):
    text: str
    label: str

@app.get("/")
def home():
    return {"message": "Sentiment Classification API is running!"}

@app.get("/status")
def status():
    return {**state, "buffered_requests": len(recent_X), "feedback_samples": len(fb_X)}

@app.get("/debug_drift")
def debug_drift():
    if len(recent_X) < 10:
        return {"error": "not enough buffered requests yet"}
    new = np.array(recent_X[-min(len(recent_X), 200):])
    frac = drift_fraction(X_ref, new)
    return {"drift_fraction": frac, "buffered": len(recent_X)}

@app.post("/predict")
def predict(review: ReviewInput, bg: BackgroundTasks):
    emb = embed_model.encode([review.text])
    recent_X.append(emb[0])
    pred = clf.predict(emb)[0]
    if len(recent_X) % 50 == 0:
        bg.add_task(maybe_heal)
    return {"review": review.text, "sentiment": pred, "model_version": state["model_version"]}

@app.post("/feedback")
def feedback(fb: Feedback):
    emb = embed_model.encode([fb.text])
    fb_X.append(emb[0]); fb_y.append(fb.label)
    return {"feedback_samples": len(fb_X)}

Writing app.py
